<a href="https://colab.research.google.com/github/goutham3010/Hospital-Patient-Flow-and-Resource-Demand-Prediction-System/blob/main/03_Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HOSPITAL PATIENT FLOW & DEMAND PREDICTION SYSTEM
## NOTEBOOK 03: DATA CLEANING & FEATURE ENGINEERING
**Workflow Phases 03 & 04** of the 17-Stage Hospital Operations Decision Support Architecture.

### Objectives:
1. Clean raw admission events and enforce chronological validity (`discharge_date >= admit_date`).
2. Aggregate discrete patient admissions into continuous daily time-series operational flow ($A_t, D_t, C_t$).
3. Construct predictive feature matrices:
   - Calendar and cyclical features (`sin_dow`, `cos_dow`, `sin_month`, `cos_month`).
   - Autoregressive lag features ($A_{t-1}, A_{t-2}, A_{t-3}, A_{t-7}, A_{t-14}$).
   - Rolling moving averages and volatility statistics (7-day, 14-day, 30-day).
   - Short-term vs. long-term surge momentum ($\Delta_{7, 30}$).
   - Department-level historical demand lags (General, ICU, HDU, NICU).
   - Active inpatient bed census ($C_{t-1}$).
4. Define the primary forecasting target: **Tomorrow's Patient Admissions ($A_{t+1}$)**.
5. Export engineered feature dataset (`data/daily_patient_flow.csv`).


In [ ]:
# 1. IMPORT LIBRARIES & MODULES
import pandas as pd
import numpy as np
import sys
import os

sys.path.append('src')
from data_loader import load_raw_dataset
from preprocessing import clean_admissions_data, aggregate_to_daily_series, engineer_patient_flow_features, get_feature_columns

print("Preprocessing modules loaded.")


In [ ]:
# 2. LOAD RAW ADMISSIONS DATA
raw_df = load_raw_dataset('admissions.csv')
print(f"Raw Admissions Count: {len(raw_df):,}")


In [ ]:
# 3. CLEANING & CHRONOLOGICAL VALIDATION
cleaned_df = clean_admissions_data(raw_df)
print(f"Validated Inpatient Records: {len(cleaned_df):,}")
print("Verifying non-negative length of stay:", (cleaned_df['los_days'] >= 0).all())


In [ ]:
# 4. TIME-SERIES AGGREGATION TO DAILY HOSPITAL FLOW
daily_df = aggregate_to_daily_series(cleaned_df)
print(f"Daily series generated across {len(daily_df):,} days ({daily_df['date'].min().date()} to {daily_df['date'].max().date()}).")
daily_df.head(5)


In [ ]:
# 5. FEATURE ENGINEERING (LAGS, ROLLING STATS & TARGETS)
featured_df = engineer_patient_flow_features(daily_df)
print(f"Feature matrix generated: {featured_df.shape[0]} days x {featured_df.shape[1]} columns.")
featured_df[['date', 'admissions_total', 'admissions_lag_1', 'admissions_roll_mean_7',
             'admissions_momentum_7_30', 'target_tomorrow_admissions']].head(5)


In [ ]:
# 6. VERIFY NO FORWARD LOOKAHEAD LEAKAGE
feature_list = get_feature_columns()
print(f"Total Engineered Predictive Features: {len(feature_list)}")
print("\nTop 15 Features:")
for f in feature_list[:15]:
    print(f" • {f}")


In [ ]:
# 7. SAVE FINAL ENGINEERED DATASET
os.makedirs('data', exist_ok=True)
output_path = 'data/daily_patient_flow.csv'
featured_df.to_csv(output_path, index=False)
print(f"Engineered dataset saved to {output_path}")
